In [28]:
import math

import torch
import torch.nn as nn

# Attention

$$Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

- transformer中tensor将batchsize放在第一个维度，因为可以同时对所有token进行处理，并不需要按照序列顺序依次处理；而RNN中将seq_len放在第一个维度，是因为RNN需要按序列顺序处理；
- 除以$\sqrt{d_k}$是因为特征维度越大，两个特征向量的点乘值越大，softmax之后分布越尖锐，注意力大部分集中在几个token上，所以除以标准差$\sqrt{d_k}$，让注意力logits值也符合均值为0，方差为1的分布
- Self-Attention：序列本身内部的token进行的
- Cross-Attention：像RNN里，decoder序列里的token对encoder里的token计算注意力，是不同序列token之间计算的注意力

In [29]:
# attention

class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super(MultiHeadAttentionBlock, self).__init__()
        self.d_model = d_model  # embedding特征大小
        self.h = h  # 头的个数
        # 确保 d_model 可以被 h 整除
        assert d_model % h == 0, "d_model 不能被 h 整除"

        self.d_k = d_model // h # 每个头特征大小
        self.w_q = nn.Linear(d_model, d_model, bias=False)  # Wq
        self.w_k = nn.Linear(d_model, d_model, bias=False)  # Wk
        self.w_v = nn.Linear(d_model, d_model, bias=False)  # Wv
        self.w_o = nn.Linear(d_model, d_model, bias=False)  # Wo
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        # 获取 d_k 值
        d_k = query.shape[-1]

        # Q乘K转置，除以根号d_k     # 为了方便计算，维度会提前进行变换
        # [batch, h, seq_len, d_k] --> [batch, h, seq_len, seq_len]
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            # 在mask为0的位置填入一个很大的负值，则在进行softmax时注意力为0
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)

        # softmax，归一化，得到注意力权重
        # [batch, h, seq_len, seq_len]
        attention_scores = attention_scores.softmax(dim=-1) # 每一行的和为1

        if dropout is not None:
            attention_scores = dropout(attention_scores)

        # 注意力权重乘 V，得到更新后的embedding
        # [batch, h, seq_len, seq_len] --> [batch, h, seq_len, d_k]
        return (attention_scores @ value), attention_scores

    def forward(self, q, k, v, mask):
        # 通过三个全连接层，得到Q,K,V矩阵
        query = self.w_q(q)     # [batch, seq_len, d_model] --> [batch, seq_len, d_model]
        key = self.w_k(k)       # [batch, seq_len, d_model] --> [batch, seq_len, d_model]
        value = self.w_v(v)     # [batch, seq_len, d_model] --> [batch, seq_len, d_model]

        # 对多头进行拆分，并变形
        # [batch, seq_len, d_model] --> [batch, seq_len, h, d_k] --> [batch, h, seq_len, d_k]
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        # 计算注意力
        x, self.attention_score = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

        # 多头合并
        # [batch, h, seq_len, d_k] --> [batch, seq_len, h, d_k] --> [batch, seq_len, d_model]
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)

        # 乘以输出层
        return self.w_o(x)

# Layer Normalization

- 如果还用batch norm的问题：序列可能很长，导致batch很小，不同batch之间的均值和方差差距很大；短序列填充了很多<pad>token，会影响正常token的均值和方差计算
- layer norm：按每个序列，分别统计所有特征的均值和方差，每个维度都定义两个可学习的参数$\gamma, \beta$来进行吸纳行变换。即对每个token独立做标准化，但使用一组全局共享的逐维缩放$\gamma$和偏移$\beta$参数。
$$y = \frac{x-E[X]}{Std[x] + \epsilon} * \gamma + \beta$$
- 经过layer norm后基本保证每个token的特征分布都是均值为0，方差为1
- 是对每个token embedding的维度进行的，不用区分训练状态和推理状态

In [30]:
# layer norm

class LayerNormalization(nn.Module):
    def __init__(self, features: int, eps: float = 1e-6) -> None:
        super(LayerNormalization, self).__init__()
        self.eps = eps
        # 可学习权重
        self.alpha = nn.Parameter(torch.ones(features))
        # 可学习偏置
        self.bias = nn.Parameter(torch.zeros(features))

    def forward(self, x):
        # x: [batch, seq_len, hidden_size]
        mean = x.mean(-1, keepdim=True)     # [batch, seq_len, 1]
        std = x.std(-1, keepdim=True)       # [batch, seq_len, 1]

        return self.alpha * (x - mean) / (std + self.eps) + self.bias

# 位置编码 Positional Encoding

$$w_i = \frac{1}{10000^{i/feature\_size}}$$
\*$w_i$是每个编码位置的sin()或cos()函数的系数，i从0开始，到feature_size-1结束，

- 每个token的位置编码都在sin或cos函数里的$w_i$前乘以自己的位置序号，$sin(t * w_i)$
- 随着编码位置的增加，$w_i$变小，意味着，三角函数的波长变长，值的变换变慢
- 偶数位置用sin函数，系数用$w_{2i}$表示，奇数位置用cos函数，系数用$w_{2i+1}$表示
- 因为注意力机制是通过q和k向量的点积来计算，对于使用sin和cos交替进行编码的两个位置向量，t和t+Δt，进行点积运算：
$[sin(w_it),cos(w_it)] \cdot [sin(w_i(t+\triangle t)),cos(w_i(t+\triangle t))] = cos(w_i \triangle t)$。最后注意力计算结果只与两个位置之间的相对位置Δt有关，而与绝对位置 t 无关

In [31]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super(PositionalEncoding, self).__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # 创建一个空的tensor
        pe = torch.zeros(seq_len, d_model)  # [seq_len, d_model]

        # 创建一个位置向量，表示每个token在序列中的位置 [0.0,1.0,2.0,...,seq_len-1]
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)

        # 计算分母
        div_term = torch.pow(10000, -torch.arange(0, d_model, 2, dtype=torch.float) / d_model) # (d_model/2)

        # 偶数位置调用sin
        pe[:, 0::2] = torch.sin(position * div_term)
        # 奇数位置调用cos
        pe[:, 1::2] = torch.cos(position * div_term)

        # 增加batch维度
        pe = pe.unsqueeze(0)    # [1, seq_len, d_model]

        # 注册位置编码为一个buffer，这个tensor不参与训练，但是会随同模型一起被保存或者迁移到GPU
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)  # 取出和当前输入长度对齐的位置编码
        return self.dropout(x)

# transformer里的encoder

- 输入：一个batch的token_id列表，经过embedding之后，再经过PE加上每个token的位置编码；
- EncoderBlock：N个EncoderBlock，每个内部是[多头注意力层 → 残差连接和层归一化] → [全连接层 → 残差连接和层归一化]；
- EncoderBlock的输入为inputs embedding张量，维度为[batch_size, seq_len, d_k]，在内部维度不变，所以可以堆叠多层EncoderBlock
- 前馈神经网络FFN：feed-forward network，定义两层全连接，第一层升维，然后relu激活，第二层重新降维。让每个token更新自身embedding
$$FFN(x) = max(0, xW_1+b_1)W_2+b_2$$

In [32]:
# FFN

class FeedForwardBlock(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super(FeedForwardBlock, self).__init__()
        self.linear_1 = nn.Linear(d_model, d_ff, bias=True)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model, bias=True)

    def forward(self, x):
        # [batch, seq_len, d_model] -> [batch, seq_len, d_ff] -> [batch, seq_len, d_model]
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

In [33]:
# Add & Norm

class ResidualConnection(nn.Module):
    def __init__(self, features: int, dropout: float) -> None:
        super(ResidualConnection, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNormalization(features)

    def forward(self, x, sublayer):
        # sublayer可以是多头注意力模块，也可以是全连接模块
        return x + self.dropout(sublayer(self.norm(x)))

In [34]:
# Encoder

class EncoderBlock(nn.Module):
    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock,
                 feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super(EncoderBlock, self).__init__()
        # 定义多头自注意力块
        self.self_attention_block = self_attention_block
        # 定义全连接块
        self.feed_forward_block = feed_forward_block
        # 定义两个 add&norm 块
        self.residual_connections = nn.ModuleList([
            ResidualConnection(features, dropout) for _ in range(2)
        ])

    def forward(self, x, src_mask):
        # 第一个残差连接，跟在多头注意力模块后
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, src_mask))
        # 第二个残差连接，跟在全连接后
        x = self.residual_connections[1](x, self.feed_forward_block)
        return x


class Encoder(nn.Module):
    def __init__(self, features: int, layers: nn.ModuleList) -> None:
        super(Encoder, self).__init__()
        # 传入EncoderBlock
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, mask):
        # 依次调用6个EncoderBlock
        for layer in self.layers:
            x = layer(x, mask)
        # 输出前进行LayerNorm
        return self.norm(x)

# Transformer 里的 Decoder

- 每个DecoderBlock内部是2个注意力模块(1个Masked Multi-Head Attention和1个Cross-Attention)和1个全连接模块；
- 第一个注意力模块masked自注意力和encoder里面一样；
- 第二个注意力模块cross-attention，K和V矩阵来自encoder的输出，Q矩阵来自decoder

In [35]:
class DecoderBlock(nn.Module):
    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock,
                 cross_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super(DecoderBlock, self).__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(3)])

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        # tgt_mask是上三角被mask掉，防止decoder时偷窥未来信息
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, tgt_mask))
        # src_mask是为了屏蔽encoder里面<pad>的无意义信息，保证注意力落在真实输入上
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connections[2](x, self.feed_forward_block)
        return x


class Decoder(nn.Module):
    def __init__(self, features: int, layers: nn.ModuleList) -> None:
        super(Decoder, self).__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

In [36]:
# 将decoder的输出投影到词库

class ProjectionLayer(nn.Module):

    def __init__(self, d_model, vocab_size) -> None:
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x) -> None:
        # (batch, seq_len, d_model) --> (batch, seq_len, vocab_size)
        return self.proj(x)

In [37]:
# embedding

class InputEmbeddings(nn.Module):
    def __init__(self, d_model, vocab_size) -> None:
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        # [batch, seq_len] -> [batch, seq_len, d_model]
        return self.embedding(x) * math.sqrt(self.d_model)

组装整个Transformer

In [38]:
class Transformer(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder,
                 src_embed: InputEmbeddings, tgt_embed: InputEmbeddings,
                 src_pos: PositionalEncoding, tgt_pos: PositionalEncoding,
                 projection_layer: ProjectionLayer) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer

    def encode(self, src, src_mask):
        # [batch, seq_len, d_model]
        src = self.src_embed(src)
        src = self.src_pos(src)
        return self.encoder(src, src_mask)

    def decode(self, encoder_output: torch.Tensor, src_mask: torch.Tensor, tgt: torch.Tensor, tgt_mask: torch.Tensor) -> torch.Tensor:
        # [batch, seq_len, d_model]
        tgt = self.tgt_embed(tgt)
        tgt = self.src_pos(tgt)
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)

    def project(self, x):
        # [batch, seq_len, vocab_size]
        return self.projection_layer(x)

In [39]:
def build_transformer(src_vocab_size: int, tgt_vocab_size: int,
                      src_seq_len: int, tgt_seq_len: int,
                      d_model: int=512, N: int=6, h: int=8, dropout: float=0.1, d_ff: int=2048) -> Transformer:
    # embedding
    src_embed = InputEmbeddings(d_model, src_vocab_size)
    tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)

    # pe
    src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
    tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)

    # encoder block
    encoder_blocks = []
    for _ in range(N):
        encoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
        encoder_block = EncoderBlock(d_model, encoder_self_attention_block, feed_forward_block, dropout)
        encoder_blocks.append(encoder_block)

    # decoder block
    decoder_blocks = []
    for _ in range(N):
        decoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
        decoder_cross_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
        decoder_block = DecoderBlock(d_model, decoder_self_attention_block, decoder_cross_attention_block, feed_forward_block, dropout)
        decoder_blocks.append(decoder_block)

    # encoder & decoder
    encoder = Encoder(d_model, nn.ModuleList(encoder_blocks))
    decoder = Decoder(d_model, nn.ModuleList(decoder_blocks))

    # projection
    projection_layer = ProjectionLayer(d_model, tgt_vocab_size)

    # transformer
    transformer = Transformer(encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, projection_layer)

    # 初始化参数
    for p in transformer.parameters():
        # 只有权重矩阵维度会大于1，进行初始化；bias是向量，一般直接设为0
        if p.dim() > 1:
            # Xavier初始化让信号在前向传播和反向传播中保持方差稳定
            nn.init.xavier_uniform_(p)

    return transformer

翻译模型训练

In [40]:
# 定义数据集
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

class TranslationDataset(Dataset):
    ## 初始化方法，读取英文和中文训练文本。然后给每个句子前后增加<bos>和<eos>。 为了防止训练时显存不足，对于长度超过限制的
    ## 句子进行过滤。
    def __init__(self, src_file, trg_file, src_tokenizer, trg_tokenizer, max_len=100):
        with open(src_file, encoding='utf-8') as f:
            src_lines = f.read().splitlines()
        with open(trg_file, encoding='utf-8') as f:
            trg_lines = f.read().splitlines()
        assert len(src_lines) == len(trg_lines)
        self.pairs = []
        self.src_tokenizer = src_tokenizer
        self.trg_tokenizer = trg_tokenizer
        index = 0
        for src, trg in zip(src_lines, trg_lines):
            index += 1
            if index % 100000 == 0:
                print(index)
            # 每个句子前边增加<bos>后边增加<eos>
            src_ids = [BOS_ID] + self.src_tokenizer(src) + [EOS_ID]
            trg_ids = [BOS_ID] + self.trg_tokenizer(trg) + [EOS_ID]
            # 只保留输入和输出序列token数同时小于max_len的训练样本。
            if len(src_ids) <= max_len and len(trg_ids) <= max_len:
                self.pairs.append((src_ids, trg_ids))  # <-- 直接保存token id序列

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_ids, trg_ids = self.pairs[idx]
        return torch.LongTensor(src_ids), torch.LongTensor(trg_ids)

    ## 对一个batch的输入和输出token序列，依照最长的序列长度，用<pad> token进行填充，确保一个batch的数据形状一致，组成一个tensor。
    @staticmethod
    def collate_fn(batch):
        src_batch, trg_batch = zip(*batch)
        src_lens = [len(x) for x in src_batch]
        trg_lens = [len(x) for x in trg_batch]
        ## 注意，Transformer里的tensor，设置batch_frist=True。
        src_pad = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=PAD_ID)
        trg_pad = nn.utils.rnn.pad_sequence(trg_batch, batch_first=True,padding_value=PAD_ID)
        return src_pad, trg_pad, src_lens, trg_lens

生成mask
- 对于encoder输入序列，在计算自注意力时，不关注<pad>token
- 对于decoder输入序列，在进行自注意力时，不关注<pad>token和当前token后面的token

In [41]:
def create_mask(src, tgt, pad_idx):
    # mask <pad> for encoder
    src_mask = (src != PAD_ID).unsqueeze(1).unsqueeze(2)    # [batch, 1, 1, src_len]    # 变成4维，因为需要能广播成[batch, head, seq_len, seq_len]形状，对应于attention的score形状
    # mask <pad> for decoder
    tgt_pad_mask = (tgt != PAD_ID).unsqueeze(1).unsqueeze(2)    # [batch, 1, 1, tgt_len]

    tgt_len = tgt.size(1)
    # decoder mask 当前token后边的token
    tgt_sub_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool() # 下三角 [tgt_len, tgt_len]
    # decoder 同时mask <pad>，以及当前token后面的token
    tgt_mask = tgt_pad_mask & tgt_sub_mask  # [batch, 1, tgt_len, tgt_len].
    return src_mask, tgt_mask

In [42]:
import sentencepiece as spm

sp_cn = spm.SentencePieceProcessor()
sp_cn.load(r"C:\Users\61075\PycharmProjects\learning\torch\runs\tokenizer\zh_bpe.model")
sp_en = spm.SentencePieceProcessor()
sp_en.load(r"C:\Users\61075\PycharmProjects\learning\torch\runs\tokenizer\en_bpe.model")

def tokenize_en(text):
    return sp_en.encode(text, out_type=int)
def tokenize_cn(text):
    return sp_cn.encode(text, out_type=int)

PAD_ID = sp_en.pad_id()  # 1
UNK_ID = sp_en.unk_id()  # 0
BOS_ID = sp_en.bos_id()  # 2
EOS_ID = sp_en.eos_id()  # 3

In [44]:
# train
def train(model, dataloader, criterion, optimizer, pad_idx):
    model.train()
    total_loss = 0
    step = 0
    log_loss = 0

    for src, tgt, src_lens, tgt_lens in dataloader:
        step += 1

        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        tgt_input = tgt[:, :-1] # 给定前面所有token
        tgt_output = tgt[:, 1:] # 预测下一个token

        src_mask, tgt_mask = create_mask(src, tgt_input, pad_idx)

        optimizer.zero_grad()
        encoder_output = model.encode(src, src_mask)
        decoder_output = model.decode(encoder_output, src_mask, tgt_input, tgt_mask)
        output = model.project(decoder_output)

        output = output.reshape(-1, output.shape[-1])
        tgt_output = tgt_output.reshape(-1)

        loss = criterion(output, tgt_output)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        log_loss += loss.item()

        if step % 100 == 0:
            avg_log_loss = log_loss / 100
            print(f"Step: {step}, avg loss: {avg_log_loss:.4f}")
            log_loss = 0

    return total_loss / len(dataloader)



if __name__ == "__main__":
    # 超参数
    SRC_VOCAB_SIZE = 16000
    TRG_VOCAB_SIZE = 16000
    SRC_SEQ_LEN = 128
    TRG_SEQ_LEN = 128
    BATCH_SIZE = 64
    NUM_EPOCHS = 10
    LR = 1e-4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 数据集加载
    train_dataset = TranslationDataset(
        src_file=r"D:\agent\torch\data\en2cn\train_en_10000.txt",
        trg_file=r"D:\agent\torch\data\en2cn\train_zh_10000.txt",
        src_tokenizer=tokenize_en,
        trg_tokenizer=tokenize_cn,
    )
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=train_dataset.collate_fn)

    # 构建模型
    model = build_transformer(SRC_VOCAB_SIZE, TRG_VOCAB_SIZE, SRC_SEQ_LEN, TRG_SEQ_LEN).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    for epoch in range(NUM_EPOCHS):
        loss = train(model, train_dataloader, criterion, optimizer, PAD_ID)
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}, loss: {loss:.4f}")

    torch.save(
        model.state_dict(),
        r'C:\Users\61075\PycharmProjects\learning\torch\runs\translation\transformer.pt'
    )

Step: 100, avg loss: 7.2462
Epoch 1/10, loss: 6.8221
Step: 100, avg loss: 5.7578
Epoch 2/10, loss: 5.7191
Step: 100, avg loss: 5.4866
Epoch 3/10, loss: 5.4603
Step: 100, avg loss: 5.2434
Epoch 4/10, loss: 5.2317
Step: 100, avg loss: 5.0209
Epoch 5/10, loss: 5.0142
Step: 100, avg loss: 4.7889
Epoch 6/10, loss: 4.7867
Step: 100, avg loss: 4.5897
Epoch 7/10, loss: 4.5808
Step: 100, avg loss: 4.3860
Epoch 8/10, loss: 4.3822
Step: 100, avg loss: 4.1778
Epoch 9/10, loss: 4.1870
Step: 100, avg loss: 3.9972
Epoch 10/10, loss: 4.0002


In [48]:
SRC_VOCAB_SIZE = 16000
TRG_VOCAB_SIZE = 16000
SRC_SEQ_LEN = 128
TRG_SEQ_LEN = 128
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model = build_transformer(SRC_VOCAB_SIZE, TRG_VOCAB_SIZE, SRC_SEQ_LEN, TRG_SEQ_LEN).to(DEVICE)
model.load_state_dict(torch.load(r'C:\Users\61075\PycharmProjects\learning\torch\runs\translation\transformer.pt', map_location=DEVICE))
model.eval()


def create_mask(src, pad_idx):
    return (src != pad_idx).unsqueeze(1).unsqueeze(2)  # (1, 1, 1, src_len)


def translate_sentence(sentence, max_len=100):
    # Tokenize and convert to IDs
    tokens = [BOS_ID] + sp_en.encode(sentence, out_type=int) + [EOS_ID]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(DEVICE)  # [1, src_len]
    src_mask = create_mask(src_tensor, PAD_ID)

    # Initialize target with <bos> token
    trg_indices = [BOS_ID]

    with torch.no_grad():
        # Encode the source sentence
        encoder_output = model.encode(src_tensor, src_mask)

        # Generate translation token by token
        for _ in range(max_len):
            trg_tensor = torch.LongTensor(trg_indices).unsqueeze(0).to(DEVICE)  # [1, current_trg_len]

            # Create target mask
            trg_mask = torch.tril(torch.ones((len(trg_indices), len(trg_indices)), device=DEVICE)).bool()
            trg_mask = trg_mask.unsqueeze(0).unsqueeze(0)  # [1, 1, trg_len, trg_len]

            # Decode
            decoder_output = model.decode(encoder_output, src_mask, trg_tensor, trg_mask)
            output = model.project(decoder_output)

            # Get the last predicted token
            pred_token = output.argmax(2)[:, -1].item()
            trg_indices.append(pred_token)

            if pred_token == EOS_ID:
                break

    # Convert token IDs to text (skip <bos> and <eos>)
    translated = sp_cn.decode(trg_indices[1:-1])
    return translated


# ---------------------#
# 4. Interactive Test
# ---------------------#
if __name__ == '__main__':
    print("Transformer Translator (type 'quit' or 'exit' to end)")
    while True:
        src_sent = input("\nEnter English sentence: ")
        if src_sent.lower() in ['quit', 'exit']:
            break
        translation = translate_sentence(src_sent)
        print(f"Chinese Translation: {translation}")

Transformer Translator (type 'quit' or 'exit' to end)
Chinese Translation: 一支队伍都在离开在一起。


KeyboardInterrupt: Interrupted by user

In [49]:
# BLEU指标评估
import sacrebleu

# 读取验证集的英文原文和中文原文(都只取前20个)
with open(r"D:\agent\torch\data\en2cn\valid_en.txt", "r", encoding="utf-8") as f:
    src_sentences = [line.strip() for _, line in zip(range(20), f)]
with open(r"D:\agent\torch\data\en2cn\valid_zh.txt", "r", encoding="utf-8") as f:
    ref_sentences = [line.strip() for _, line in zip(range(20), f)]

# 检查长度是否匹配
assert len(src_sentences) == len(ref_sentences), "源语言和参考翻译句子数不匹配"

# 生成翻译
hypotheses = []
for i, src in enumerate(src_sentences):
    print(f"translating {i+1}/{len(src_sentences)}...")
    translation = translate_sentence(src)
    print(ref_sentences[i], translation)
    hypotheses.append(translation.strip())  # 去掉首和尾的空格后添加

# 计算BLEU
bleu = sacrebleu.corpus_bleu(hypotheses, [ref_sentences], tokenize='zh')
print("\n========== BLEU Evaluation Result ==========")
print(f"BLEU Score: {bleu.score:.2f}")

translating 1/20...
你们觉得我们看起来够年轻溜进高中吗？ 一想到你那个人你那个子的什么?
translating 2/20...
嗨，亲爱的。你现在肯定忙着开会呢。 一张纸。我一张给你。
translating 3/20...
因为你想在进养老院前娶妻生子。 一心只想想另一方面她。
translating 4/20...
我就一天24小时都得在她眼皮子底下。 一整天都都在想电话我就觉得觉得觉得觉得。
translating 5/20...
找条牢靠的链子或者别的什么固定住这些灯。 一手拿着交一手交交交钱。
translating 6/20...
为了不让别的父母经历我的遭遇。 一想到你都在想马上想兴奋。
translating 7/20...
我要去赴约会，必须学跳舞。现在就学。 一想到要要自己上就觉得觉得觉得。
translating 8/20...
有时候我们信任的人替我们做了这样的选择。 一想到要要离开我就觉得觉得觉得觉得。
translating 9/20...
好吧。那么，我想现在能做的有限。 一无所知。我我太太难了。
translating 10/20...
我尊重这点，并且会不惜一切保护隐私不被侵犯。 一想到我,我我离开我就觉得觉得觉得觉得觉得觉得。
translating 11/20...
太奇怪了。- 我们出去吧。 一把刀。我不,我更多。
translating 12/20...
所以在调查了有微量血迹的门把手之后， 一小块人就会放人,
translating 13/20...
如果我们找不到她，她就只有两周可活了。 一手交钱,一手交交钱。
translating 14/20...
但是如果我们能摒弃前嫌 一想到,我我离开我说我
translating 15/20...
如果你还想做球队经理的话你必须得做这个。 一手拿着钱一手交交钱。
translating 16/20...
好吗？你挥着你的小翅膀飞下来然后告诉他我不知道。 一把钥匙就想钥匙。
translating 17/20...
因为你要知道不管你在保护谁等你们一起进了号子 一心一意都在想想想起来?
translating 18/20...
你们还可以看到剧场台阶保存得十分完好； 一心只想想离开哭
translating 19/20...
你得照顾他的孩子，我是说在干掉他